## Creates pattern_support table

In [1]:
import sqlite3
import pandas as pd

## Configuration

In [2]:
# verbimustrite andmebaas
PATTERN_DB = "../example_data/verb_patterns.db"

PATTERNS_TABLE = "patterns"
VERB_MATCHES_TABLE = "verb_matches"
PATTERNS_META_TABLE = "patterns_meta"

NEW_TABLE_NAME = "pattern_support"

## Connect to db

In [3]:
con = sqlite3.connect(PATTERN_DB)
cur = con.cursor()

## Workflow

In [5]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=NEW_TABLE_NAME))

cur.execute("""
CREATE TABLE {new_table} AS
SELECT
    pm.pat_id,
    verb_word,
    verb_compound,
    phrase_case,
    adp,
    inf_verb,
    verb_match_count AS verb_occurrence_count,
    phrase_count AS absolute_support,
    CAST(phrase_count AS REAL) / CAST(verb_match_count AS REAL) * 100 AS relative_support
FROM
(
    SELECT 
        pat.pat_id as pat_id,
        verb_word,
        verb_compound,
        phrase_case,
        adp,
        inf_verb,
        count(*) AS verb_match_count
    FROM
        {tbl1} as vm
    INNER JOIN
        {tbl2} as pat
    ON
        pat.pat_id = vm.pat_id
    GROUP BY
        pat.pat_id
) as tbl
INNER JOIN
    {tbl3} as pm
ON
    tbl.pat_id = pm.pat_id
ORDER BY
    relative_support DESC
""".format(new_table=NEW_TABLE_NAME, tbl1=VERB_MATCHES_TABLE, tbl2=PATTERNS_TABLE, tbl3=PATTERNS_META_TABLE))

CPU times: user 6.75 ms, sys: 2.36 ms, total: 9.11 ms
Wall time: 11.8 ms


In [4]:
query = """SELECT * FROM sqlite_master WHERE type='table'"""
source = pd.read_sql_query(query, con)
source

,type,name,tbl_name,rootpage,sql
0,table,patterns,patterns,2,CREATE TABLE patterns\n (pat_id INTEGER PRI...
1,table,semantic_annotations,semantic_annotations,148,CREATE TABLE semantic_annotations\n (patter...
2,table,patterns_meta,patterns_meta,333,CREATE TABLE patterns_meta (\n pat_id INTEG...
3,table,verb_phrase_matches,verb_phrase_matches,335,CREATE TABLE verb_phrase_matches (\n pat_id...
4,table,pattern_support,pattern_support,336,"CREATE TABLE pattern_support(\n pat_id INT,\n..."
5,table,verb_matches,verb_matches,340,"CREATE TABLE verb_matches(\n pat_id INT,\n h..."


In [12]:
con.close()